In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ (her notebook'un ilk hücresi) ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"   # private repo ise token kullanılır
BRANCH   = "main"                                           # kendi branch'inizde çalışıyorsanız değiştirin
TOKEN    = userdata.get("GH_TOKEN")                         # Colab > Secrets > GH_TOKEN (sadece private repo)
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
if "/content/repo/src" not in sys.path:      # editable install is only seen after a kernel restart
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()          # config/config.yaml + DATA_ROOT
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 01 — Exploration: descriptive statistics

Stage 1.1. This notebook contains no business logic; it only calls `recsys.analysis.explore`.

* Input: stage **`loaded`** (LATEST), validated against the loaded schemas.
* Output: one **`exploration`** version holding all six tables, each validated against its schema,
  with a manifest that references the `loaded` version used.
* `exploratory_targets` is for exploration only. The model target is recomputed in 1.9 from
  train interactions only.

In [ ]:
from recsys.analysis.explore import (
    EXPLORATION_SCHEMAS,
    basic_counts,
    column_type_table,
    exploratory_targets,
    per_entity_counts,
    rating_counts,
    top_items,
)
from recsys.contracts import LOADED_INTERACTIONS_SCHEMA, LOADED_RECIPES_SCHEMA, validate
from recsys.io import ArtifactStore

TOP_N = 20            # stage 1.1 spec: top 20 ingredients and tags
MAX_CATEGORIES = 10   # only used for columns without a declared type

store = ArtifactStore.from_config(CFG)
loaded = store.load("loaded")
recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)
interactions = validate(loaded["interactions"], LOADED_INTERACTIONS_SCHEMA)
print("input: loaded", loaded.version, "| recipes", recipes.shape, "| interactions", interactions.shape)

## 1. Column types

In [ ]:
column_types = column_type_table(
    {"recipes": recipes, "interactions": interactions}, max_categories=MAX_CATEGORIES
)
column_types

**Comment draft** (English, two to three sentences; replace every `{...}` placeholder with the value shown in the output above, do not type numbers from memory):

> The two tables contain columns of every type class: identifiers and `rating` are categorical, durations, counts and nutrition values are continuous, `tags`, `steps` and `ingredients` are lists, and names, descriptions and reviews are free text. `description` has {column_types.n_missing[recipes, description]} missing values and `review` has {column_types.n_missing[interactions, review]}.

**Comment:**



## 2. Basic counts

In [ ]:
basic = basic_counts(recipes, interactions)
basic.T

**Comment draft** (English, two to three sentences; replace every `{...}` placeholder with the value shown in the output above, do not type numbers from memory):

> The catalogue has {basic_counts.n_recipes} recipes from {basic_counts.n_contributors} contributors and {basic_counts.n_interactions} interactions by {basic_counts.n_users} users, between {basic_counts.interactions_date_min} and {basic_counts.interactions_date_max}. {basic_counts.n_recipes_without_interactions} recipes have no interaction and stay only in the recommendation pool; {basic_counts.n_duplicate_interactions} rows are exact duplicates (handled in 1.3).

**Comment:**



## 3. Interactions per recipe and per user

In [ ]:
per_entity = per_entity_counts(interactions, recipes)
per_entity

**Comment draft** (English, two to three sentences; replace every `{...}` placeholder with the value shown in the output above, do not type numbers from memory):

> Activity is very uneven: the median recipe has {per_entity_counts.median[recipe]} interactions while the maximum is {per_entity_counts.max[recipe]}, and {per_entity_counts.pct_one[user]} of users wrote exactly one review. This sparsity matters for personalisation, because most users have too little history to build a profile from.

**Comment:**



## 4. Rating values

In [ ]:
ratings = rating_counts(interactions)
ratings

**Comment draft** (English, two to three sentences; replace every `{...}` placeholder with the value shown in the output above, do not type numbers from memory):

> Ratings are strongly skewed towards the top of the scale: {rating_counts.share_of_rated[5]} of rated interactions give the maximum score. {rating_counts.count[0]} interactions have rating = 0, which on Food.com means no stars were given; they are treated as missing in 1.3, not as the lowest score. Because of the imbalance, the majority class is only a baseline.

**Comment:**



## 5. Most frequent ingredients and tags

In [ ]:
top = top_items(recipes, n=TOP_N)
top

**Comment draft** (English, two to three sentences; replace every `{...}` placeholder with the value shown in the output above, do not type numbers from memory):

> The most common ingredient is {top_items.item[ingredient, rank 1]} ({top_items.share_of_recipes[ingredient, rank 1]} of recipes) and the most common tag is {top_items.item[tag, rank 1]}. Basic pantry items dominate the ingredient list, so they carry little information for similarity; raw strings still contain variants that are merged in 1.5.

**Comment:**



## 6. Exploratory rating targets (exploration only)

In [ ]:
targets = exploratory_targets(recipes, interactions)
display(targets.head())
targets[["n_interactions", "n_ratings", "avg_rating"]].describe()

**Comment draft** (English, two to three sentences; replace every `{...}` placeholder with the value shown in the output above, do not type numbers from memory):

> {exploratory_targets.has_rating.sum} recipes have at least one non-zero rating; the others have no `avg_rating` and are excluded from training but kept for recommendation. The mean of `avg_rating` is {exploratory_targets.avg_rating.mean}. These values use all interactions and are not model targets; those are recomputed from train data in 1.9.

**Comment:**



## 7. Save stage `exploration`

In [ ]:
exploration = store.save(
    "exploration",
    {
        "column_types": column_types,
        "basic_counts": basic,
        "per_entity_counts": per_entity,
        "rating_counts": ratings,
        "top_items": top,
        "exploratory_targets": targets,
    },
    inputs=[loaded],
    params={"top_n": TOP_N, "max_categories": MAX_CATEGORIES},
    schemas=EXPLORATION_SCHEMAS,
)
print("saved:", exploration.stage, exploration.version, "->", exploration.path)
check = store.load("exploration")  # re-read with sha256 verification
{name: f["n_rows"] for name, f in check.manifest["files"].items()}